# PathSnap Hyperparameter Optimisation (Bayesian Tuning with Optuna)
This notebook runs an automated Bayesian hyperparameter search (using Optuna) to optimize model performance (maximizing Macro F1-Score and Top-1 Accuracy) for the map-matching classification pipeline.

In [2]:
import os
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import cross_val_score
import optuna

# Suppress minor warnings during optimization logs
optuna.logging.set_verbosity(optuna.logging.WARNING)
print("Optuna and ML libraries imported successfully.")

Optuna and ML libraries imported successfully.


c:\Users\PRADYUMNA\PathSnap-MapMatching\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
# Simulated feature data placeholder (will use Laptop 1's preprocessed features when ready)
np.random.seed(42)
X_dummy = np.random.rand(500, 10)
y_dummy = np.random.randint(0, 5, size=500)

def objective(trial):
    """
    Optuna objective function to tune Random Forest hyperparameters:
    - n_estimators
    - max_depth
    - min_samples_split
    """
    params = {
        'n_estimators': trial.suggest_int('n_estimators', 20, 200),
        'max_depth': trial.suggest_int('max_depth', 3, 20),
        'min_samples_split': trial.suggest_int('min_samples_split', 2, 10),
        'random_state': 42
    }
    
    clf = RandomForestClassifier(**params)
    
    # Evaluate using 5-fold cross-validation (maximizing Macro F1)
    scores = cross_val_score(clf, X_dummy, y_dummy, cv=3, scoring='f1_macro')
    return scores.mean()

print("Objective function defined.")

Objective function defined.


In [4]:
# Create Optuna study to maximize cross-validation score
study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=15)

print("Hyperparameter Optimization Complete!")
print(f"Best Cross-Validation Macro F1: {study.best_value:.4f}")
print("Best Hyperparameters found:")
for key, value in study.best_params.items():
    print(f"  - {key}: {value}")

# Save best parameters to results folder for model training integration
os.makedirs("../results", exist_ok=True)
best_params_df = pd.DataFrame([study.best_params])
best_params_df.to_csv("../results/best_hyperparameters.csv", index=False)
print("\nSaved best parameters to results/best_hyperparameters.csv")

Hyperparameter Optimization Complete!
Best Cross-Validation Macro F1: 0.2349
Best Hyperparameters found:
  - n_estimators: 117
  - max_depth: 11
  - min_samples_split: 9

Saved best parameters to results/best_hyperparameters.csv
